### 1. Impor Pustaka & Memuat Dataset
Melakukan impor pustaka utama (`pandas`, `numpy`, `matplotlib`, `seaborn`), membaca file dataset `Social_media_impact_on_life.csv`, serta menampilkan 5 baris pertama dan informasi dasar dari dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('Social_media_impact_on_life.csv')

print("=== 5 Baris Pertama Dataset ===")
print(df.head())

print("\n=== Informasi Dataset ===")
print(df.info())

### 2. Informasi Detail & Pengecekan Missing Value
Menampilkan informasi lengkap struktur DataFrame (tipe data, penggunaan memori), dimensi dataset (`shape`), rincian tipe data tiap kolom, serta jumlah nilai yang kosong (*missing values*).

In [ ]:
df.info()

print("Jumlah baris dan kolom:", df.shape)

print("\nTipe data:")
print(df.dtypes)

print("\nJumlah nilai kosong:")
print(df.isnull().sum())

### 3. Statistik Deskriptif
Menghasilkan ringkasan statistik deskriptif (seperti nilai rata-rata, standar deviasi, nilai minimum, kuartil, dan maksimum) untuk kolom numerik, serta menampilkan total baris dan kolom.

In [ ]:
print("=== Statistik Deskriptif ===")
display(df.describe())

print(f"\nJumlah baris: {df.shape[0]}")
print(f"Jumlah kolom: {df.shape[1]}")

### 4. Analisis Missing Value Lebih Lanjut
Menghitung secara spesifik jumlah nilai kosong pada setiap kolom dan menyaring kolom mana saja yang memiliki *missing value* lebih dari 0.

In [ ]:
missing_values = df.isnull().sum()

print("Jumlah Missing Value:")
print(missing_values)

print("\nKolom dengan Missing Value:")
print(missing_values[missing_values > 0])

### 5. Visualisasi Box Plot untuk Fitur Numerik
Membuat visualisasi *box plot* untuk setiap kolom bertipe numerik guna mendeteksi sebaran data serta potensi adanya pencilan (*outliers*).

In [ ]:
numeric_features = df.select_dtypes(include="number").columns

for feature in numeric_features:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=df[feature])
    plt.title(f'Box Plot of {feature}')
    plt.show()

### 6. Pengecekan Baris Duplikat
Memeriksa apakah terdapat baris data yang terduplikat di dalam dataset serta menampilkan jumlah dan baris duplikat tersebut jika ada.

In [ ]:
duplicates = df.duplicated()

print("Jumlah baris duplikat:", duplicates.sum())
print("\nBaris duplikat:")
display(df[duplicates])

### 7. Identifikasi Kolom Kategorikal
Menampilkan daftar kolom kategorikal/boolean serta jumlah kategori unik pada masing-masing kolom tersebut.

In [ ]:
category_features = df.select_dtypes(include=["object", "category", "bool"]).columns

print("Kolom kategorikal:")
print(category_features.tolist())

print("\nJumlah kategori unik setiap kolom:")
print(df[category_features].nunique())

### 8. Prapemrosesan Data (Preprocessing Pipeline)
Menyiapkan *pipeline* prapemrosesan menggunakan `ColumnTransformer` untuk menangani *missing value* (imputasi median untuk numerik, modus untuk kategorikal), melakukan *standard scaling* pada data numerik, serta *one-hot encoding* pada data kategorikal.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

target_col = 'Overall_Impact'
X = df.drop(columns=['Student_ID', target_col])
y = df[target_col]

num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)

### 9. Visualisasi Target & Pembagian Data (Data Splitting)
Melihat distribusi kelas pada variabel target (`Overall_Impact`) menggunakan diagram batang, lalu membagi dataset menjadi data latih (*training set*) dan data uji (*test set*) dengan rasio 80:20.

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='Overall_Impact', palette='viridis')
plt.title('Distribusi Variabel Target (Overall_Impact)')
plt.xlabel('Overall Impact')
plt.ylabel('Jumlah Responden')
plt.show()

### 10. Pelatihan Model (Model Training)
Membangun *pipeline* akhir yang menggabungkan *preprocessor* dan algoritma klasifikasi *Random Forest*, lalu melatih model menggunakan data latih (*X_train*, *y_train*).

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Jumlah seluruh data:", len(df))
print("Jumlah data latih:", len(X_train))
print("Jumlah data uji:", len(X_test))

### 11. Evaluasi Model & Visualisasi Confusion Matrix
Melakukan prediksi pada data uji, menampilkan laporan performa (*classification report*), serta memvisualisasikan matriks kebingungan (*confusion matrix*) menggunakan `seaborn heatmap`.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

target_col = 'Overall_Impact'
X = df.drop(columns=[target_col])
if 'Student_ID' in X.columns:
    X = X.drop(columns=['Student_ID'])

X_train = X_train.drop(columns=['Student_ID'], errors='ignore')
X_test = X_test.drop(columns=['Student_ID'], errors='ignore')

num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category', 'bool', 'string']).columns.tolist()

X_train[cat_cols] = X_train[cat_cols].astype(str)
X_test[cat_cols] = X_test[cat_cols].astype(str)

num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))